# Linear RRF Weight Model

This notebook learns query-variant weights for weighted reciprocal rank fusion (RRF).

It uses recorded query variants from `output_with_agents_uc1.csv` and `output_with_agents_uc2.csv`, so it does not call the LLM agents.

Workflow:
- Split each use case and process level into two parts.
- Use the first part to learn nonnegative linear weights for the query variants.
- Apply the learned weights to the second part.
- Report held-out precision, recall, and F1.

## Setup

This cell imports the libraries, finds the project root, and adds it to `sys.path` so local modules such as `main.py` and `retrieval/retrieval_bm25.py` can be imported from inside the notebook folder.

In [7]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

project_root = Path.cwd()
while project_root.name != "Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval" and project_root.parent != project_root:
    project_root = project_root.parent

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from main import clean_text
from retrieval.retrieval_bm25 import Query, build_bm25_index, load_corpus

print(f"Working directory: {Path.cwd()}")

Working directory: /Users/mareklorenz/Development/Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval


## Model Definition

Each query variant produces its own BM25 ranked list. For every candidate document, the notebook builds one feature per variant:

`feature_variant = 1 / (RRF_K + rank_variant)`

If a document is not retrieved by a variant, that feature is `0`. The linear model learns how much each variant's RRF feature should matter.

The training target is binary: `1` if the candidate document is in the gold standard for the query, otherwise `0`. A logistic regression is trained on the first split, negative coefficients are clipped to zero, and the remaining coefficients are normalized so they sum to `1`. Those normalized coefficients become the learned RRF weights.

In [ ]:
USE_CASES = ["uc1", "uc2"]
LEVELS = ["process", "subprocess", "task"]
RRF_K = 60
RANDOM_STATE = 42

VARIANTS = [
    ("baseline", "query"),
    ("legal_terminology_rewrite", "legal_terminology_rewrite"),
    ("regulatory_compliance_query", "regulatory_compliance_query"),
    ("contract_clause_query", "contract_clause_query"),
    ("risk_scenario_query", "risk_scenario_query"),
]

LEVEL_TO_GS_SUFFIX = {
    "process": "process_level",
    "subprocess": "subprocess_level",
    "task": "event_level",
}

LEVEL_TO_TOP_K = {
    "process": 100,
    "subprocess": 30,
    "task": 15,
}


def corpus_path_for(use_case):
    return Path(f"regulatory_relevance4process/SOTA_NLP_LIR/input_ranking/{use_case}/Input_corpus_{use_case}.xlsx")


def gold_path_for(use_case, level):
    suffix = LEVEL_TO_GS_SUFFIX[level]
    return Path(
        f"regulatory_relevance4process/SOTA_NLP_LIR/output_ranking_input_eval/{use_case}/gold_standard/gs_{use_case}_{suffix}.xlsx"
    )


def recorded_path_for(use_case):
    return Path(f"output_with_agents_{use_case}.csv")


def load_records(use_case):
    path = recorded_path_for(use_case)
    if not path.exists():
        raise FileNotFoundError(f"Missing recorded query variants: {path}")

    df = pd.read_csv(path)
    df["level"] = df["level"].astype(str).str.lower()
    df["query_clean"] = df["query"].apply(clean_text)
    return df


def split_queries(queries):
    queries = list(queries)
    if len(queries) == 1:
        return queries, queries, "single_query_reused"

    rng = np.random.default_rng(RANDOM_STATE)
    indices = np.arange(len(queries))
    rng.shuffle(indices)
    split_at = max(1, len(indices) // 2)
    train_indices = set(indices[:split_at])
    train_queries = [query for idx, query in enumerate(queries) if idx in train_indices]
    test_queries = [query for idx, query in enumerate(queries) if idx not in train_indices]
    return train_queries, test_queries, "random_half_split"


def variant_text(row, column):
    value = row[column]
    if pd.isna(value) or str(value).strip() == "":
        return clean_text(row["query"])
    return clean_text(value)


def rank_variant_lists(bm25_index, row, top_k):
    ranked_lists = {}
    for variant_name, column in VARIANTS:
        ranked_lists[variant_name] = bm25_index.rank(Query(text=variant_text(row, column)), top_k=top_k)
    return ranked_lists


def rrf_feature_rows(ranked_lists):
    doc_features = {}
    for variant_index, (variant_name, _) in enumerate(VARIANTS):
        for result in ranked_lists[variant_name]:
            features = doc_features.setdefault(result.document.doc_id, np.zeros(len(VARIANTS), dtype=float))
            features[variant_index] = 1.0 / (RRF_K + result.rank)
    return doc_features


def build_examples(records_df, bm25_index, gold_lookup, query_subset, top_k):
    examples = []
    labels = []
    groups = []

    for query in query_subset:
        row = records_df[records_df["query_clean"] == query].iloc[0]
        ranked_lists = rank_variant_lists(bm25_index, row, top_k=top_k)
        doc_features = rrf_feature_rows(ranked_lists)
        positives = gold_lookup.get(query, set())

        for doc_id, features in doc_features.items():
            examples.append(features)
            labels.append(1 if doc_id in positives else 0)
            groups.append((query, doc_id))

    return np.array(examples), np.array(labels), groups


def learn_weights(x_train, y_train):
    if len(np.unique(y_train)) < 2:
        return np.ones(len(VARIANTS)) / len(VARIANTS), "equal_weights_fallback"

    model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
    model.fit(x_train, y_train)
    raw_weights = np.maximum(model.coef_[0], 0.0)

    if raw_weights.sum() == 0:
        return np.ones(len(VARIANTS)) / len(VARIANTS), "equal_weights_fallback"

    return raw_weights / raw_weights.sum(), "linear_logistic_coefficients"


def rank_with_weights(records_df, bm25_index, query_subset, top_k, weights):
    predictions = {}
    for query in query_subset:
        row = records_df[records_df["query_clean"] == query].iloc[0]
        ranked_lists = rank_variant_lists(bm25_index, row, top_k=top_k)
        doc_features = rrf_feature_rows(ranked_lists)
        scored_docs = [
            (doc_id, float(np.dot(features, weights)))
            for doc_id, features in doc_features.items()
        ]
        scored_docs.sort(key=lambda item: item[1], reverse=True)
        predictions[query] = {doc_id for doc_id, _ in scored_docs[:top_k]}
    return predictions


def evaluate_predictions(predictions, gold_lookup):
    tp = fp = fn = 0
    for query, predicted in predictions.items():
        gold = gold_lookup.get(query, set())
        tp += len(predicted & gold)
        fp += len(predicted - gold)
        fn += len(gold - predicted)

    precision = tp / (tp + fp) if tp + fp else np.nan
    recall = tp / (tp + fn) if tp + fn else np.nan
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else np.nan
    return {
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def run_weight_learning(use_case, level, override_weights=None, override_source=None):
    top_k = LEVEL_TO_TOP_K[level]
    records_df = load_records(use_case)
    level_records = records_df[records_df["level"] == level].copy()

    gold_df = pd.read_excel(gold_path_for(use_case, level))
    gold_df["query_clean"] = gold_df["query"].apply(clean_text)
    documents = load_corpus(str(corpus_path_for(use_case)))
    bm25_index = build_bm25_index(str(corpus_path_for(use_case)))
    doc_id_by_text = {clean_text(document.text): document.doc_id for document in documents}

    gold_lookup = {}
    for query, group in gold_df.groupby("query_clean"):
        gold_doc_ids = {
            doc_id_by_text[clean_text(rel_text)]
            for rel_text in group["rel_text"].tolist()
            if clean_text(rel_text) in doc_id_by_text
        }
        gold_lookup[query] = gold_doc_ids

    queries = [query for query in level_records["query_clean"].tolist() if query in gold_lookup]
    train_queries, test_queries, split_note = split_queries(queries)

    if override_weights is None:
        x_train, y_train, _ = build_examples(level_records, bm25_index, gold_lookup, train_queries, top_k)
        weights, weight_source = learn_weights(x_train, y_train)
    else:
        weights = np.array(override_weights, dtype=float)
        weight_source = override_source or "reused_weights"
        split_note = f"{split_note}_with_reused_weights"

    predictions = rank_with_weights(level_records, bm25_index, test_queries, top_k, weights)
    metrics = evaluate_predictions(predictions, gold_lookup)

    weight_row = {
        "use_case": use_case,
        "level": level,
        "split_note": split_note,
        "weight_source": weight_source,
        "train_queries": len(train_queries) if override_weights is None else 0,
        "test_queries": len(test_queries),
    }
    for (variant_name, _), weight in zip(VARIANTS, weights):
        weight_row[variant_name] = weight

    metric_row = {
        "use_case": use_case,
        "level": level,
        "split_note": split_note,
        "train_queries": len(train_queries) if override_weights is None else 0,
        "test_queries": len(test_queries),
        **metrics,
    }
    return weight_row, metric_row

## Train Weights

This cell runs the experiment for every `use_case` and process `level`.

For subprocess and task levels:
- Recorded query variants are loaded from `output_with_agents_<use_case>.csv`.
- Queries are split into train and test halves with a fixed random seed.
- BM25 retrieves candidates for each query variant.
- Logistic regression learns the variant weights on the train half.

For process level:
- No separate process-level weights are learned, because there is only one process query.
- The subprocess weights from the same use case are reused for process-level evaluation.
- The process row is marked with `reused_subprocess_weights`.

In [9]:
weight_rows = []
metric_rows = []

for use_case in USE_CASES:
    level_results = {}

    subprocess_weight_row, subprocess_metric_row = run_weight_learning(use_case, "subprocess")
    subprocess_weights = [subprocess_weight_row[variant_name] for variant_name, _ in VARIANTS]
    level_results["subprocess"] = (subprocess_weight_row, subprocess_metric_row)

    process_weight_row, process_metric_row = run_weight_learning(
        use_case,
        "process",
        override_weights=subprocess_weights,
        override_source="reused_subprocess_weights",
    )
    level_results["process"] = (process_weight_row, process_metric_row)

    task_weight_row, task_metric_row = run_weight_learning(use_case, "task")
    level_results["task"] = (task_weight_row, task_metric_row)

    for level in LEVELS:
        weight_row, metric_row = level_results[level]
        weight_rows.append(weight_row)
        metric_rows.append(metric_row)

weights_df = pd.DataFrame(weight_rows)
metrics_df = pd.DataFrame(metric_rows)

weights_df

,use_case,level,split_note,weight_source,train_queries,test_queries,baseline,legal_terminology_rewrite,regulatory_compliance_query,contract_clause_query,risk_scenario_query
0,uc1,process,single_query_reused_with_reused_weights,reused_subprocess_weights,0,1,0.166278,0.000000,0.237166,0.334244,0.262312
1,uc1,subprocess,random_half_split,linear_logistic_coefficients,3,4,0.166278,0.000000,0.237166,0.334244,0.262312
2,uc1,task,random_half_split,linear_logistic_coefficients,14,15,0.432559,0.000000,0.000000,0.000000,0.567441
3,uc2,process,single_query_reused_with_reused_weights,reused_subprocess_weights,0,1,0.064816,0.329225,0.248223,0.349635,0.008101
4,uc2,subprocess,random_half_split,linear_logistic_coefficients,3,4,0.064816,0.329225,0.248223,0.349635,0.008101
5,uc2,task,random_half_split,linear_logistic_coefficients,9,10,0.292944,0.077133,0.326251,0.038122,0.265551


## Held-Out Evaluation

The learned weights are applied to the test half only. For each test query, documents are ranked by the weighted sum of their RRF features:

`score(document) = sum(weight_variant * feature_variant)`

The top `k` documents are treated as predicted relevant, using the same level-specific cutoffs as the retrieval pipeline: `100` for process, `30` for subprocess, and `15` for task/event.

Precision, recall, and F1 are then computed against the gold-standard relevant documents for the held-out queries.

In [10]:
display_columns = [
    "use_case",
    "level",
    "split_note",
    "train_queries",
    "test_queries",
    "true_positives",
    "false_positives",
    "false_negatives",
    "precision",
    "recall",
    "f1",
]

metrics_display = metrics_df[display_columns].copy()
metrics_display[["precision", "recall", "f1"]] = metrics_display[["precision", "recall", "f1"]].round(3)
metrics_display

,use_case,level,split_note,train_queries,test_queries,true_positives,false_positives,false_negatives,precision,recall,f1
0,uc1,process,single_query_reused_with_reused_weights,0,1,18,82,31,0.180,0.367,0.242
1,uc1,subprocess,random_half_split,3,4,9,111,29,0.075,0.237,0.114
2,uc1,task,random_half_split,14,15,13,212,60,0.058,0.178,0.087
3,uc2,process,single_query_reused_with_reused_weights,0,1,24,76,7,0.240,0.774,0.366
4,uc2,subprocess,random_half_split,3,4,12,108,28,0.100,0.300,0.150
5,uc2,task,random_half_split,9,10,18,132,47,0.120,0.277,0.167


## Export Results

The final cell writes two sheets to `linear_rrf_weight_model_results.xlsx`:

- `learned_weights`: one row per use case and process level, containing the learned query-variant weights.
- `heldout_metrics`: precision, recall, F1, and confusion counts on the held-out split.

In [11]:
output_path = Path("linear_rrf_weight_model_results.xlsx")
with pd.ExcelWriter(output_path) as writer:
    weights_df.to_excel(writer, sheet_name="learned_weights", index=False)
    metrics_df.to_excel(writer, sheet_name="heldout_metrics", index=False)

print(f"Saved {output_path}")

Saved linear_rrf_weight_model_results.xlsx
